# YOLO11n 船舶检测模块消融训练

本 Notebook 一次只训练一个实验。先完成仓库、数据和模型审计，再由你手动运行“开始训练”单元格。正式训练总目标始终是 150 轮；若启用人工检查点，第 80 轮保存完成后暂停，并使用 `last.pt` 无缝恢复。

In [ ]:
# 顶部配置：每次只修改一个实验名，避免连续误触多个正式实验。
EXPERIMENT_NAME = "yolo11n-dd"
DATA_YAML_RELATIVE = "data.yaml"  # 若云端存在多个 YAML，请明确填写正式基线使用的相对路径。
BASELINE_RESULTS_CSV = ""  # 可填写 Drive 中已有 baseline results.csv 的绝对路径。
MAX_EPOCHS = 150
SCREEN_EPOCH = 80
PAUSE_AT_SCREEN_EPOCH = True
REPO_URL = "https://github.com/HoverdZ/ship-yolo.git"
REPO_REF = "experiment/yolo11n-crossconv-dd-cgfm"
REPO_ROOT = "/content/ship-yolo"


In [ ]:
# 挂载 Google Drive。不会修改 Drive 上的原始数据集。
from google.colab import drive
drive.mount("/content/drive")


## 安全获取私有仓库

Token 只存在于当前内存和 Git 子进程的临时 HTTP Header；不会打印、写入 remote URL、保存到 Drive 或提交到 Git。

In [ ]:
import base64
import os
import subprocess
from getpass import getpass
from pathlib import Path

github_token = getpass("请输入GitHub Token：")
credential = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.extraHeader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credential}",
})
try:
    if (Path(REPO_ROOT) / ".git").is_dir():
        subprocess.run(["git", "-C", REPO_ROOT, "fetch", "origin", REPO_REF], check=True, env=git_env)
        subprocess.run(["git", "-C", REPO_ROOT, "switch", REPO_REF], check=True)
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only", "origin", REPO_REF], check=True, env=git_env)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, REPO_ROOT], check=True, env=git_env)
finally:
    github_token = ""
    credential = ""
    git_env.pop("GIT_CONFIG_VALUE_0", None)

remote_url = subprocess.run(["git", "-C", REPO_ROOT, "remote", "get-url", "origin"], check=True, text=True, capture_output=True).stdout.strip()
assert remote_url == REPO_URL, f"remote URL 异常：{remote_url}"
print("仓库 remote：", remote_url)


In [ ]:
# 使用仓库自己的可编辑安装，避免自动升级到不匹配的 Ultralytics 最新版。
%pip install -e /content/ship-yolo


In [ ]:
# 导入仓库训练辅助代码并打印完整环境。
import sys
sys.path.insert(0, REPO_ROOT)
from colab.train_yolo11n_module_ablation import (
    TrainingConfig,
    copy_dataset_to_local,
    create_local_data_yaml,
    print_environment,
    print_training_plan,
    resume_training,
    start_training,
    sync_training_script_to_drive,
)
commit = subprocess.run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip()
print_environment(commit)
sync_training_script_to_drive()


## 复制并核对数据集

使用多线程 `shutil.copyfile` 从 Drive 递归复制到 Colab 本地磁盘。这里只核对当前云端副本自身的文件、图片、标签和 YAML 清单，不要求它与其他本地数据集具有固定相同数量。

In [ ]:
inventory = copy_dataset_to_local()
data_yaml = create_local_data_yaml(DATA_YAML_RELATIVE)


In [ ]:
# 训练前审计：本单元格只构建模型并打印计划，不启动训练。
config = TrainingConfig(
    experiment_name=EXPERIMENT_NAME,
    data_yaml_relative=DATA_YAML_RELATIVE,
    baseline_results_csv=BASELINE_RESULTS_CSV,
    epochs=MAX_EPOCHS,
    screen_epoch=SCREEN_EPOCH,
    pause_at_screen_epoch=PAUSE_AT_SCREEN_EPOCH,
)
print_training_plan(config, data_yaml)


## 开始训练（手动运行）

下面直接在当前 Python 进程调用 Ultralytics `model.train()`，保留官方逐 epoch、逐 batch 实时输出。第 80 轮验证和保存完成后 callback 会暂停，并生成当前模型与 baseline 的曲线和摘要；不会用单个指标自动淘汰模型。

In [ ]:
# 正式训练入口：只有明确准备好后才运行此单元格。
results = start_training(config, data_yaml)


## 从第 80 轮继续到第 150 轮（独立手动运行）

人工决定继续后，把路径改为本次 run 的 `last.pt`。这里不再注册暂停 callback，并通过 `resume=True` 恢复优化器、学习率调度器和 epoch；不要使用 `best.pt` 重新微调。

In [ ]:
LAST_PT = f"/content/drive/MyDrive/ship_detection/runs/{EXPERIMENT_NAME}/weights/last.pt"
# 人工确认后取消下一行注释：
# resume_results = resume_training(LAST_PT)
